# In-the-Wild Inference

Run the trained affordance model on phone photos / natural captures.

Workflow:
1. Drop images into `data/in_the_wild/` (jpg / png).
2. Run the **Setup** cell once — loads the model into memory.
3. Use **Predict single** to inspect one image interactively.
4. Use **Predict folder** to batch-process everything in the input folder.

Predictions are written to `reports/in_the_wild_predictions/`.

## Setup — load model once

In [ ]:
import sys, os
from pathlib import Path

# Project root on sys.path so we can import config, models, scripts
PROJECT_ROOT = Path(os.getcwd()).resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.predict import (
    load_model, load_and_prepare, predict_image, render_prediction,
    IMG_EXTS,
)
import torch
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

INPUT_DIR  = PROJECT_ROOT / "data" / "in_the_wild"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "in_the_wild_predictions"
CHECKPOINT = PROJECT_ROOT / "checkpoints" / "best.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT}")
print(f"Input folder: {INPUT_DIR}")
print(f"Output folder: {OUTPUT_DIR}")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

backbone, decoder, normalize = load_model(str(CHECKPOINT), DEVICE)
print("Model loaded.")

## List available images

In [ ]:
images = sorted(p for p in INPUT_DIR.iterdir()
                if p.suffix.lower() in IMG_EXTS and not p.name.startswith("."))
if not images:
    print(f"No images in {INPUT_DIR} yet. Drop some .jpg / .png files there.")
else:
    print(f"Found {len(images)} images:")
    for i, p in enumerate(images):
        print(f"  [{i}] {p.name}")

## Predict single — pick an image by index and inspect inline

In [ ]:
INDEX = 0    # change this to inspect a different image

img_path = images[INDEX]
print(f"Running on: {img_path.name}")

rgb_np = load_and_prepare(img_path)
pred_prob, pred_norm = predict_image(rgb_np, backbone, decoder, DEVICE, normalize)

out_path = OUTPUT_DIR / f"{img_path.stem}_prediction.png"
render_prediction(rgb_np, pred_prob, pred_norm, img_path.stem, out_path)
print(f"Saved {out_path}")
display(Image(str(out_path)))

## Quick text summary — which affordances did the model see?

Prints the max sigmoid probability for each of the 7 channels. Anything above ~0.5 indicates the model believes that affordance exists somewhere in the image.

In [ ]:
from config import AFFORDANCE_CLASSES

summary = sorted(
    [(c, float(pred_prob[i].max()), float(pred_prob[i].mean()))
     for i, c in enumerate(AFFORDANCE_CLASSES)],
    key=lambda r: -r[1],
)
print(f"Per-class signal for {img_path.name}:\n")
print(f"  {'class':12s}  {'max_prob':>8s}  {'mean_prob':>9s}")
for cls, mx, mn in summary:
    bar = '#' * int(mx * 20)
    print(f"  {cls:12s}  {mx:>8.3f}  {mn:>9.4f}  {bar}")

## Predict folder — batch process every image in `data/in_the_wild/`

In [ ]:
results = []
for img_path in images:
    try:
        rgb_np = load_and_prepare(img_path)
        pred_prob, pred_norm = predict_image(rgb_np, backbone, decoder, DEVICE, normalize)
        out_path = OUTPUT_DIR / f"{img_path.stem}_prediction.png"
        render_prediction(rgb_np, pred_prob, pred_norm, img_path.stem, out_path)
        results.append((img_path.name, out_path.name))
        print(f"  OK   {img_path.name}")
    except Exception as e:
        print(f"  FAIL {img_path.name}  ({e})")

print(f"\nProcessed {len(results)} images. Output in {OUTPUT_DIR}")